# 01 — Preprocessing

**Primary:** Tuan Wei  
**Support:** Tianyi Qin

RQ: **Among Melbourne entire homes and apartments with valid nightly prices, do location and amenities improve high-price prediction beyond property size alone, and which attributes are most useful?**

Purpose: inspect the official data, define the eligible cohort, collect quantified evidence for six preprocessing candidates, choose three, measure their impact, and export one stable dataset for the rest of the group.

## Official A2 data source\n\nThis notebook automatically downloads the **16 June 2026 original Melbourne Detailed Listings** file directly from Inside Airbnb if `data/listings.csv` is missing. It uses the detailed `listings.csv.gz` source, decompresses it to `data/listings.csv`, and records `data/SOURCE_METADATA.json`.\n\nThis implements the teaching-team amendment: do **not** use the cleaned/modified Assignment 1 dataset.

## 1. Imports and data path

In [ ]:
from pathlib import Path\nimport ast\nimport json\nimport math\nimport re\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nRANDOM_STATE = 42\n\n# Locate repository root whether the notebook is launched from the repo root\n# or from notebooks/.\nREPO_ROOT = Path('..').resolve() if Path('../src').exists() else Path('.').resolve()\nif str(REPO_ROOT) not in sys.path:\n    sys.path.insert(0, str(REPO_ROOT))\n\nfrom src.data_source import ensure_melbourne_listings, MELBOURNE_LISTINGS_GZ_URL\n\nDATA_PATH = ensure_melbourne_listings(REPO_ROOT / 'data' / 'listings.csv')\nprint('Official source:', MELBOURNE_LISTINGS_GZ_URL)\nprint('Using:', DATA_PATH.resolve())

## A2 dataset amendment guard\n\nThe teaching team clarified that A2 must use the **original dataset downloaded directly from Inside Airbnb** and must **not** use the cleaned/modified A1 dataset. The next cell imports the team's A1 reusable logic and checks that `data/listings.csv` is not the known 29-column A1 teaching file.

In [ ]:
from src.a1_reuse import (\n    validate_a2_source,\n    add_clean_price,\n    add_amenity_count,\n    add_bathrooms_numeric,\n)

## 2. Load raw data

Keep `df_raw` unchanged. All before/after counts should be reproducible from it.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)\nvalidate_a2_source(df_raw)\nprint('Raw shape:', df_raw.shape)\nprint('A2 source guard: passed (does not match the known A1 curated schema)')\ndisplay(df_raw.head())

## 3. Initial audit

In [ ]:
audit = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True)
}).sort_values(["missing_pct", "missing_n"], ascending=False)

display(audit.head(50))

## 4. Inspect RQ-relevant fields

In [ ]:
rq_columns = [
    "id", "price", "room_type", "property_type",
    "latitude", "longitude", "neighbourhood_cleansed",
    "accommodates", "bedrooms", "beds", "bathrooms_text",
    "amenities"
]

present = [c for c in rq_columns if c in df_raw.columns]
absent = [c for c in rq_columns if c not in df_raw.columns]

print("Present:", present)
print("Absent:", absent)
display(df_raw[present].head())

## 5. Define the eligible cohort

The RQ is restricted to **Entire home/apt** listings with a valid positive nightly price. Treat this as cohort construction, not automatically as one of the three rubric preprocessing tasks.

In [ ]:
if not {'room_type', 'price'}.issubset(df_raw.columns):\n    raise KeyError('room_type and price are required for the agreed RQ.')\n\nn_raw = len(df_raw)\n\ndf = df_raw.loc[df_raw['room_type'].eq('Entire home/apt')].copy()\nn_entire = len(df)\n\n# Reuse the corrected A1 price-cleaning logic on the ORIGINAL A2 source data.\ndf = add_clean_price(df, source='price', target='price_clean')\nvalid_price = (\n    df['price_clean'].notna()\n    & np.isfinite(df['price_clean'])\n    & df['price_clean'].gt(0)\n)\ndf = df.loc[valid_price].copy()\nn_valid = len(df)\n\ncohort_summary = pd.DataFrame({\n    'stage': [\n        'Raw listings',\n        'Entire home/apt',\n        'Entire home/apt + valid positive price',\n    ],\n    'rows': [n_raw, n_entire, n_valid],\n})\ncohort_summary['retained_pct_of_raw'] = (\n    cohort_summary['rows'] / n_raw * 100\n).round(2)\n\ndisplay(cohort_summary)

## 6. Price distribution — descriptive only

The final target threshold must follow the group contract: **training-sample 75th percentile**. Do not use the full-data Q75 below as the final classification threshold.

In [ ]:
display(
    df["price_clean"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

## 7. Candidate preprocessing evidence

Rubric: name at least **six candidates**, select **three**, and justify the selected three over the others with actual dataset numbers.

Recommended candidates to investigate:
1. Missing-data strategy for bedrooms/beds/bathrooms.
2. Engineer numeric bathrooms from `bathrooms_text`.
3. Engineer distance from Melbourne CBD from latitude/longitude.
4. Engineer amenity count / amenity indicators.
5. Consolidate rare property types.
6. Encode/consolidate neighbourhood.
7. Size-feature scaling/consistency checks if useful downstream.

You only need six; keep the six that are most defensible for this dataset.

In [ ]:
candidate_cols = [c for c in [
    "bedrooms", "beds", "bathrooms_text",
    "latitude", "longitude", "amenities",
    "property_type", "neighbourhood_cleansed",
    "accommodates"
] if c in df.columns]

candidate_audit = pd.DataFrame({
    "dtype": df[candidate_cols].dtypes.astype(str),
    "missing_n": df[candidate_cols].isna().sum(),
    "missing_pct": (df[candidate_cols].isna().mean() * 100).round(2),
    "n_unique": df[candidate_cols].nunique(dropna=True),
})
display(candidate_audit)

### Category diagnostics

In [ ]:
if "property_type" in df:
    property_counts = df["property_type"].value_counts(dropna=False)
    display(property_counts.head(30))
    print("Property types:", property_counts.size)
    print("Property types with <10 listings:", int((property_counts < 10).sum()))

if "neighbourhood_cleansed" in df:
    neighbourhood_counts = df["neighbourhood_cleansed"].value_counts(dropna=False)
    display(neighbourhood_counts.head(30))
    print("Neighbourhood categories:", neighbourhood_counts.size)

size_cols = [c for c in ["accommodates", "bedrooms", "beds"] if c in df]
if size_cols:
    display(df[size_cols].describe())

## 8. Reusable feature-engineering helpers

In [ ]:
def parse_bathrooms(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan

def parse_amenities(value):
    if pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set)):
        return list(value)

    s = str(value).strip()

    # Inside Airbnb commonly stores a JSON-like list of strings.
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return list(parsed)
    except Exception:
        pass

    if s in {"", "[]", "{}"}:
        return []

    return [x.strip().strip('"').strip("'") for x in s.strip("[]{}").split(",") if x.strip()]

def haversine_km(lat, lon, ref_lat=-37.8136, ref_lon=144.9631):
    lat1 = np.radians(pd.to_numeric(lat, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon, errors="coerce"))
    lat2 = np.radians(ref_lat)
    lon2 = np.radians(ref_lon)

    dlat = lat1 - lat2
    dlon = lon1 - lon2

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

## 9. Candidate derived variables

These cells collect evidence. They do **not** automatically mean these are the three final preprocessing tasks.

In [ ]:
if "bathrooms_text" in df:
    df["bathrooms_num_candidate"] = df["bathrooms_text"].apply(parse_bathrooms)

if "amenities" in df:
    df["amenities_list_candidate"] = df["amenities"].apply(parse_amenities)
    df["amenity_count_candidate"] = df["amenities_list_candidate"].str.len()

if {"latitude", "longitude"}.issubset(df.columns):
    df["distance_cbd_km_candidate"] = haversine_km(
        df["latitude"], df["longitude"]
    )

candidate_derived = [c for c in [
    "bathrooms_num_candidate",
    "amenity_count_candidate",
    "distance_cbd_km_candidate",
] if c in df.columns]

if candidate_derived:
    display(df[candidate_derived].describe())

## 10. Quantified decision table for the six candidates

Fill this table only with values produced by this notebook. Do not use generic reasons.

In [ ]:
candidate_decisions = pd.DataFrame([
    {"candidate": "Missing-data strategy for size variables", "dataset_evidence": None, "selected": None, "reason": None},
    {"candidate": "Numeric bathrooms from bathrooms_text", "dataset_evidence": None, "selected": None, "reason": None},
    {"candidate": "Distance from CBD", "dataset_evidence": None, "selected": None, "reason": None},
    {"candidate": "Amenity count / amenity indicators", "dataset_evidence": None, "selected": None, "reason": None},
    {"candidate": "Property-type consolidation", "dataset_evidence": None, "selected": None, "reason": None},
    {"candidate": "Neighbourhood encoding/consolidation", "dataset_evidence": None, "selected": None, "reason": None},
])

display(candidate_decisions)

## 11. Final three preprocessing tasks

After the group selects three tasks, implement them below. For each, preserve:
- the exact dataset number supporting selection;
- an alternative considered;
- measurable before/after impact;
- limitation to carry into the report.

Do not invent any values.

In [ ]:
selected_preprocessing = {
    "task_1": None,
    "task_2": None,
    "task_3": None,
}
selected_preprocessing

## 12. Export one stable handoff dataset

Only enable after the final three preprocessing steps have been implemented and checked. Preserve `id` and `price_clean` for later evaluation/traceability.

In [ ]:
# Example:
# output_candidates = [Path("../data/processed_listings.csv"), Path("data/processed_listings.csv")]
# OUTPUT_PATH = output_candidates[0] if output_candidates[0].parent.exists() else output_candidates[1]
# df.to_csv(OUTPUT_PATH, index=False)
# print("Saved:", OUTPUT_PATH.resolve())